In [8]:
# 07_model_quotes.ipynb

# ── Cell 1: Imports & paths ───────────────────────────────────────────────
import polars as pl
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# ── Cell 2: Load data ─────────────────────────────────────────────────────
df_quotes_daily = pl.read_parquet(PROCESSED_DIR / "quotes_fleet_daily.parquet")
print(df_quotes_daily.schema)
print(df_quotes_daily.shape)
df_quotes_daily.head(10)

Schema({'quote_date': Date, 'daily_quote_count': UInt32, 'daily_bookings': Int64, 'daily_conversion_rate': Float64, 'median_yield': Float64, 'month': Int8, 'day_of_week': Int8, 'year': Int32, 'quarter': Int8})
(731, 9)


quote_date,daily_quote_count,daily_bookings,daily_conversion_rate,median_yield,month,day_of_week,year,quarter
date,u32,i64,f64,f64,i8,i8,i32,i8
2024-01-01,42,5,0.119048,12561.291943,1,1,2024,1
2024-01-02,164,16,0.097561,9501.291549,1,2,2024,1
2024-01-03,199,16,0.080402,10819.249482,1,3,2024,1
2024-01-04,137,10,0.072993,10467.692308,1,4,2024,1
2024-01-05,148,33,0.222973,9468.447876,1,5,2024,1
2024-01-06,64,9,0.140625,9465.883467,1,6,2024,1
2024-01-07,72,5,0.069444,9275.573111,1,7,2024,1
2024-01-08,168,13,0.077381,9660.001246,1,1,2024,1
2024-01-09,152,19,0.125,9273.91385,1,2,2024,1


In [10]:
# ── Cell 3: Prophet prep ──────────────────────────────────────────────────
df_prophet = (
    df_quotes_daily
    .select([
        pl.col("quote_date").alias("ds"),
        pl.col("daily_quote_count").alias("y")
    ])
    .sort("ds")
    .to_pandas()
)

df_prophet["ds"] = pd.to_datetime(df_prophet["ds"])

train = df_prophet[df_prophet["ds"] < "2026-01-01"]
actuals_jan = df_prophet[df_prophet["ds"] >= "2026-01-01"]  # will be empty — forecast only

print(f"Train rows      : {len(train)}")
print(f"Jan 2026 actuals: {len(actuals_jan)} days")
print(f"Train y — mean  : {train['y'].mean():.1f}, std: {train['y'].std():.1f}")

Train rows      : 731
Jan 2026 actuals: 0 days
Train y — mean  : 64.8, std: 33.8


In [11]:
# ── Cell 4: Fit & evaluate Prophet ───────────────────────────────────────
def fit_evaluate_prophet_quotes(df_train, df_actuals=None, label="quote_count"):
    """
    Fit Prophet on daily quote counts.
    Evaluate on df_actuals if provided (Jan 2026 held-out).
    Returns: model, forecast_df, metrics_dict
    """
    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode="multiplicative",  # quote volume scales with base level
        changepoint_prior_scale=0.05,       # conservative — avoid overfitting 2yr series
    )
    m.fit(df_train)

    # Forecast through Jan 2026 (31 days beyond train end)
    future = m.make_future_dataframe(periods=31, freq="D")
    forecast = m.predict(future)

    metrics = {}
    if df_actuals is not None and len(df_actuals) > 0:
        merged = df_actuals.merge(
            forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
            on="ds", how="inner"
        )
        mae  = mean_absolute_error(merged["y"], merged["yhat"])
        rmse = np.sqrt(mean_squared_error(merged["y"], merged["yhat"]))
        baseline_mae = mean_absolute_error(merged["y"], [df_train["y"].mean()] * len(merged))
        metrics = {
            "mae": round(mae, 2),
            "rmse": round(rmse, 2),
            "baseline_mae": round(baseline_mae, 2),
            "vs_baseline_pct": round((mae - baseline_mae) / baseline_mae * 100, 1)
        }
        print(f"\n{label} — Prophet Evaluation on Jan 2026:")
        print(f"  MAE  = {mae:.2f}  (baseline: {baseline_mae:.2f})")
        print(f"  RMSE = {rmse:.2f}")
        print(f"  vs baseline: {metrics['vs_baseline_pct']:+.1f}%")
    else:
        print("No Jan 2026 actuals available — forecast only.")

    return m, forecast, metrics

m_quotes, forecast_quotes, metrics_quotes = fit_evaluate_prophet_quotes(
    train, actuals_jan, label="QuoteCount"
)

01:08:57 - cmdstanpy - INFO - Chain [1] start processing
01:08:57 - cmdstanpy - INFO - Chain [1] done processing


No Jan 2026 actuals available — forecast only.


In [13]:
# ── Cell 5: Plot — forecast vs actuals ───────────────────────────────────
jan_forecast = forecast_quotes[forecast_quotes["ds"] >= "2026-01-01"].copy()
jan_forecast["ds"] = pd.to_datetime(jan_forecast["ds"])

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        "Daily Quote Count — Full Series + Jan 2026 Forecast",
        "Jan 2026 Forecast Detail (with confidence interval)"
    ],
    vertical_spacing=0.12
)

# Full series
fig.add_trace(go.Scatter(
    x=df_prophet["ds"], y=df_prophet["y"],
    mode="markers", marker=dict(size=3, color="steelblue", opacity=0.5),
    name="Actuals"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=forecast_quotes["ds"], y=forecast_quotes["yhat"],
    mode="lines", line=dict(color="crimson", width=1.5),
    name="Prophet forecast"
), row=1, col=1)

# Jan 2026 detail
fig.add_trace(go.Scatter(
    x=jan_forecast["ds"].tolist() + jan_forecast["ds"].tolist()[::-1],
    y=jan_forecast["yhat_upper"].tolist() + jan_forecast["yhat_lower"].tolist()[::-1],
    fill="toself", fillcolor="rgba(220,50,50,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="95% CI", showlegend=True
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=jan_forecast["ds"], y=jan_forecast["yhat"],
    mode="lines+markers", line=dict(color="crimson", width=2),
    marker=dict(size=6), name="Jan 2026 forecast"
), row=2, col=1)

if len(actuals_jan) > 0:
    fig.add_trace(go.Scatter(
        x=actuals_jan["ds"], y=actuals_jan["y"],
        mode="markers", marker=dict(size=8, color="steelblue", symbol="circle"),
        name="Jan 2026 actuals"
    ), row=2, col=1)

fig.update_layout(
    width=1200, height=700,
    title="07 — Quote Count Forecast (Prophet)",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.08, font=dict(size=10)),
    margin=dict(l=50, r=50, t=50, b=50)
)
fig.update_yaxes(title_text="Daily Quote Count", row=1, col=1)
fig.update_yaxes(title_text="Daily Quote Count", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()
fig.write_html("./07_quotes_forecast.html")
print("Saved → ./07_quotes_forecast.html")

Saved → ./07_quotes_forecast.html


In [14]:
# ── Cell 6: Extract Jan 2026 forecast & save ─────────────────────────────
jan2026_quotes_forecast = (
    pl.from_pandas(jan_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]])
    .rename({"ds": "date", "yhat": "forecasted_quotes"})
    .with_columns([
        pl.col("date").cast(pl.Date),
        pl.col("forecasted_quotes").round(2),
        pl.col("yhat_lower").round(2),
        pl.col("yhat_upper").round(2),
    ])
    .sort("date")
)

print(jan2026_quotes_forecast)
print(f"\nMean forecasted quotes/day: {jan2026_quotes_forecast['forecasted_quotes'].mean():.1f}")
print(f"Min: {jan2026_quotes_forecast['forecasted_quotes'].min():.1f}  "
      f"Max: {jan2026_quotes_forecast['forecasted_quotes'].max():.1f}")

jan2026_quotes_forecast.write_parquet(PROCESSED_DIR / "jan2026_quotes_forecast.parquet")
print("\n✅ Saved → data/processed/jan2026_quotes_forecast.parquet")

shape: (31, 4)
┌────────────┬───────────────────┬────────────┬────────────┐
│ date       ┆ forecasted_quotes ┆ yhat_lower ┆ yhat_upper │
│ ---        ┆ ---               ┆ ---        ┆ ---        │
│ date       ┆ f64               ┆ f64        ┆ f64        │
╞════════════╪═══════════════════╪════════════╪════════════╡
│ 2026-01-01 ┆ 77.41             ┆ 55.08      ┆ 100.42     │
│ 2026-01-02 ┆ 71.19             ┆ 47.38      ┆ 93.95      │
│ 2026-01-03 ┆ 37.26             ┆ 15.1       ┆ 59.45      │
│ 2026-01-04 ┆ 36.97             ┆ 14.01      ┆ 57.98      │
│ 2026-01-05 ┆ 94.17             ┆ 72.19      ┆ 117.3      │
│ …          ┆ …                 ┆ …          ┆ …          │
│ 2026-01-27 ┆ 98.41             ┆ 76.4       ┆ 121.86     │
│ 2026-01-28 ┆ 92.17             ┆ 71.15      ┆ 115.6      │
│ 2026-01-29 ┆ 81.41             ┆ 58.73      ┆ 103.39     │
│ 2026-01-30 ┆ 71.09             ┆ 49.39      ┆ 95.43      │
│ 2026-01-31 ┆ 33.39             ┆ 10.84      ┆ 55.66      │
└────────

In [ ]:
# # Check what Jan 2024 and Jan 2025 actually averaged
# jan_check = (
#     df_quotes_daily
#     .filter(pl.col("quote_date").dt.month() == 1)
#     .group_by(pl.col("quote_date").dt.year().alias("year"))
#     .agg(pl.col("daily_quote_count").mean().alias("jan_avg"))
#     .sort("year")
# )
# print(jan_check)

shape: (2, 2)
┌──────┬────────────┐
│ year ┆ jan_avg    │
│ ---  ┆ ---        │
│ i32  ┆ f64        │
╞══════╪════════════╡
│ 2024 ┆ 117.096774 │
│ 2025 ┆ 78.741935  │
└──────┴────────────┘
